In [1]:
import requests
from bs4 import BeautifulSoup
import urllib3
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#Fonction qui permet de faire la scraping de la page où on a récupéré les données
def scrape_page(url, session):
    data = []

    try:
        response = session.get(url, verify=False, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Erreur sur {url} : {e}")
        return data

    soup = BeautifulSoup(response.content, "lxml")

    # Trouver les tableaux imbriqués
    nested_tables = soup.select("center table td table")

    for table in nested_tables:
        rows = table.find_all("tr")

        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all("td")]

            if len(cols) == 3 and all(cols):
                fr, mg, en = cols

                if fr.lower() != "français":
                    data.append({
                        "Francais": fr,
                        "Malagasy": mg,
                        "Anglais": en,
                    })

    return data



#  Désactiver warnings SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#  Session
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

#Scaping de tous les liens où on peut récuperer des traductions
import requests
from bs4 import BeautifulSoup
import urllib3


BASE_URL = "https://motmalgache.org"

url = "https://motmalgache.org/bins/contextLists?ctxt=craft&lang=fr"
response = requests.get(url, verify=False)

soup = BeautifulSoup(response.content, "lxml")

links = []

# tableau principal
center = soup.find("center")
main_table = center.find("table")

# récupérer les colonnes
row = main_table.find("tr")
tds = row.find_all("td")

# première colonne = liens
left_td = tds[0]

# tableau imbriqué des liens
links_table = left_td.find("table")

#  récupérer les <a>
for a in links_table.find_all("a", href=True):
    href = a["href"]

    # gérer les URLs relatives
    full_url = BASE_URL + href if href.startswith("/") else href

    links.append(full_url)

print(f"{len(links)} liens trouvés")


#Récupération des données en scrapant tous les liens obtenus précédemment
all_data = []

for link in links:
    page_data = scrape_page(link, session)
    all_data.extend(page_data)

df = pd.DataFrame(all_data)

#COnstruction du CSV
columns = ["Francais", "Malagasy", "Anglais"]
df = df[[col for col in columns if col in df.columns]]
df = df.dropna()

# supprimer doublons
df = df.drop_duplicates()
df["Francais"] = df["Francais"].str.lower().str.strip()
df["Malagasy"] = df["Malagasy"].str.lower().str.strip()
df["Anglais"] = df["Anglais"].str.lower().str.strip()
df.to_csv("dictionnaire.csv", index=False, encoding="utf-8-sig")

121 liens trouvés


In [3]:
import pandas as pd
import os
#from django.conf import settings
import requests
from bs4 import BeautifulSoup
import urllib3

#CSV_PATH = os.path.join(settings.BASE_DIR, "translator", "data", "dictionnaire.csv")
df = pd.read_csv("dictionnaire.csv")

# dictionnaire rapide
dict_fr = {
    row["Francais"]: {
        "mg": row["Malagasy"],
        "en": row["Anglais"]
    }
    for _, row in df.iterrows()
}

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#Fonction qui appelle l'API pour traduire en anglais (fallback)
def translate_word_to_english_fallback(word, source="fr", target="en"):
    url = "https://api.mymemory.translated.net/get"

    params = {
        "q": word,
        "langpair": f"{source}|{target}"
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()

        return data["responseData"]["translatedText"]

    except Exception as e:
        print("API error:", e)
        return None

#Fonction qui appelle l'api et Scraping des résultas venant de l'appel à l'api de teny malagasy
def translate_teny_malagasy(word):
    url = "https://www.tenymalagasy.org/bins/teny2"

    payload = {"w": word}

    response = requests.post(
        url,
        data=payload,
        verify=False,
        timeout=10
    )

    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, "lxml")

    results = []

    # CIBLER UNIQUEMENT td.main
    main_cells = soup.select("td.main")

    for cell in main_cells:
        row = cell.find_parent("tr")
        if not row:
            continue

        cols = row.find_all("td")

        cleaned_row = []

        for col in cols:
            # récupérer uniquement les liens dans le td
            links = col.find_all("a")

            if links:
                text = ", ".join(
                    a.get_text(strip=True)
                    for a in links
                    if a.get_text(strip=True)
                )
            else:
                text = col.get_text(strip=True)

            # ignorer texte vide
            if text and text.strip():
                cleaned_row.append(text)

        # garder seulement lignes valides
        if len(cleaned_row) >= 2:
            results.append(cleaned_row)

    return results

#Fonction sui regroupe les résultats en utilisant l'api de teny malagasy
def get_mg_result(word):
    result = translate_teny_malagasy(word)

    if not result:
        return None

    words = [item[0] for item in result if item and len(item) > 0]

    # enlever doublons
    words = list(set(words))

    return ", ".join(words) if words else None

#Fonction qui orchestre la traduction
def translate_word(word):
    word = word.lower().strip()

    # LOCAL : matching avec le dataset local
    if word in dict_fr:
        return {
            "mg": dict_fr[word]["mg"],
            "en": dict_fr[word]["en"],
            "source": "local"
        }

    #Fallback si les données locales ne sont pas suffisantes
    en = translate_word_to_english_fallback(word, "fr", "en")
    mg = get_mg_result(word)

    return {
        "mg": mg if mg else "introuvable",
        "en": en if en else "not found"
    }

In [4]:
print(translate_word("bleu"))

{'mg': 'manamanga, mihamanga, herin-javona, manga, ngerona, amanga, bole', 'en': 'blue'}
